In [1]:
# Biblioteca
import arcpy
import os
from arcpy.sa import * 
import copy

# Geodatabase
gdb = r"F:\Projetos_ArcGIS_Gama\Lavagem\Lavagem.gdb"

# Projeto Atual
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.activeMap  # Mapa Ativo no ArcGIS Pro

In [5]:
# Exportar Camadas para Dataset Local - Necessário apenas se não tiver feito antes!

In [11]:
# Caminho do GDB e do Feature Dataset
feature_dataset = "Lavagem"
destino = f"{gdb}\\{feature_dataset}"

# Camadas de entrada
camadas = [
    "Via_por_AOI_v4",
    "Via_por_AOI_v4_Sul_Oeste",
    "Quadras_Label_Mapas_de_Calor",
    "Quadras_Label_Mapas_de_Calor_Sul_Oeste"
]

# Loop de exportação
for nome_camada in camadas:
    camada = mapa.listLayers(nome_camada)[0]
    saida = f"{destino}\\{nome_camada}"

    arcpy.conversion.FeatureClassToFeatureClass(
        in_features=camada,
        out_path=destino,
        out_name=nome_camada
    )

    print(f"Camada '{nome_camada}' exportada com sucesso.")

print("Processo finalizado.")

Camada 'Via_por_AOI_v4' exportada com sucesso.
Camada 'Via_por_AOI_v4_Sul_Oeste' exportada com sucesso.
Camada 'Quadras_Label_Mapas_de_Calor' exportada com sucesso.
Camada 'Quadras_Label_Mapas_de_Calor_Sul_Oeste' exportada com sucesso.
Processo finalizado.


In [6]:
# Filtro para 10 anos

In [3]:
input_layer = "BDGIS.Ordens_Servico"
anos = range(2015, 2026)

# lista para adicionar os Rasters no Contents ao final do processo
rasters_gerados = []

for ano in anos:
    
    print(f"\n🔄 Processando ano {ano}...")
    
    # Criar filtro
    sql_filter = f"""
    DATAEXECUCAOINICIO between '{ano}-01-01 00:00:00' and '{ano}-12-31 23:59:59'
    AND DESCRICAORA in ('Aguas Claras', 'Agua Quente', 'Arniqueira', 'Brazlandia', 'Ceilandia', 'Ceilandia II',
    'Gama', 'Recanto das Emas', 'Riacho Fundo', 'Riacho Fundo II', 'Samambaia', 'Santa Maria', 'Taguatinga')
    AND MOTIVONAOEXECUCAO = 0
    AND DESCSITUACAOOS = 'Baixada'
    AND SERVAPROPRIADO in ('8102008021075', '8102008021117', '8102008021118', '8102008021119', '8102008021120', '8400000032145', '8400108033354')
    """
    
    filtered_layer = f"OS_filtradas_{ano}"
    
    arcpy.MakeFeatureLayer_management(input_layer, filtered_layer, sql_filter)
    
    # Gerar Kernel
    out_raster_name = f"Lavagem_{ano}"
    out_raster_path = os.path.join(gdb, out_raster_name)
    
    kd = KernelDensity(
        in_features=filtered_layer,
        population_field=None,
        cell_size=5,
        search_radius=200,
        area_unit_scale_factor="HECTARES",
        out_cell_values="DENSITIES",
        method="PLANAR"
    )
    
    kd.save(out_raster_path)
    
    print(f"✅ Kernel {ano} gerado com sucesso.")

    # Salvar na Lista para adicionar no Contents
    rasters_gerados.append(out_raster_path)
    
print("✅ Todos os anos processados com sucesso.")

# Adicionando Rasters no Contents
for raster in rasters_gerados:
    mapa.addDataFromPath(raster)

# Excluindo Raster Redundante do Contents
for lyr in mapa.listLayers():
    if lyr.name == "kd":
        mapa.removeLayer(lyr)
        break


🔄 Processando ano 2015...
✅ Kernel 2015 gerado com sucesso.

🔄 Processando ano 2016...
✅ Kernel 2016 gerado com sucesso.

🔄 Processando ano 2017...
✅ Kernel 2017 gerado com sucesso.

🔄 Processando ano 2018...
✅ Kernel 2018 gerado com sucesso.

🔄 Processando ano 2019...
✅ Kernel 2019 gerado com sucesso.

🔄 Processando ano 2020...
✅ Kernel 2020 gerado com sucesso.

🔄 Processando ano 2021...
✅ Kernel 2021 gerado com sucesso.

🔄 Processando ano 2022...
✅ Kernel 2022 gerado com sucesso.

🔄 Processando ano 2023...
✅ Kernel 2023 gerado com sucesso.

🔄 Processando ano 2024...
✅ Kernel 2024 gerado com sucesso.

🔄 Processando ano 2025...
✅ Kernel 2025 gerado com sucesso.
✅ Todos os anos processados com sucesso.


In [7]:
# Cortando Rasters por RA

In [8]:
aoi_fc = "AOI_Sul_Oeste_v4"
name_field = "RA_Python"

# Novo gdb para salvar os dados por RA
gdb = r"F:\Projetos_ArcGIS_Gama\Lavagem\Rasters_por_RA.gdb"

# Lista para adicionar os rasters no Contents ao final do processo
rasters_gerados = []

anos = range(2015, 2026)

# Processamento
for ano in anos:
    
    print(f"\n🔄 Processando ano {ano}")
    
    # Raster já gerado anteriormente
    input_raster = f"Lavagem_{ano}"
    
    with arcpy.da.SearchCursor(aoi_fc, ["SHAPE@", name_field]) as cursor:
        for geom, nome in cursor:
            
            # Nome estruturado
            out_name = f"Lavagem_{ano}_{nome}"
            out_path = os.path.join(gdb, out_name)
            
            # Recorte do raster
            out_extract = ExtractByMask(input_raster, geom)
            out_extract.save(out_path)

            # Salvar na lista para adicionar no Contents depois
            rasters_gerados.append(out_path)

print("✅ Todos os anos processados com sucesso.")

# Adicionando Rasters Cortados no Contents
for raster in rasters_gerados:
    mapa.addDataFromPath(raster)

# Excluindo Rasters Originais do Contents
for lyr in mapa.listLayers():
    if (lyr.name.startswith("Lavagem_") and lyr.name.count("_") == 1):
        mapa.removeLayer(lyr)


🔄 Processando ano 2015

🔄 Processando ano 2016

🔄 Processando ano 2017

🔄 Processando ano 2018

🔄 Processando ano 2019

🔄 Processando ano 2020

🔄 Processando ano 2021

🔄 Processando ano 2022

🔄 Processando ano 2023

🔄 Processando ano 2024

🔄 Processando ano 2025
✅ Todos os anos processados com sucesso.


In [16]:
# Atribuindo Simbologia - Caso fique com muitas casas decimais, você deverá corrigir! 

In [22]:
for lyr in mapa.listLayers():
    
    # Filtrar apenas rasters que começam com Lavagem_2023
    if lyr.isRasterLayer and lyr.name.startswith("Lavagem_2023"):
        
        print(f"Aplicando simbologia em: {lyr.name}")
        
        # Obter simbologia
        sym = lyr.symbology
        
        # Garantir que possui colorizer
        if hasattr(sym, "colorizer"):
            
            # Converter para Raster Classify
            sym.updateColorizer('RasterClassifyColorizer')
            
            # Método de classificação
            sym.colorizer.classificationMethod = "NaturalBreaks"
            
            # Número de classes
            sym.colorizer.breakCount = 10
            
            # Rampa de cores
            sym.colorizer.colorRamp = aprx.listColorRamps("Condition Number")[0]
            
            # Aplicar simbologia inicial
            lyr.symbology = sym
            
            # ALTERAÇÃO VIA CIM
            
            # Obter definição CIM
            cim_lyr = lyr.getDefinition('V3')
            
            # PRIMEIRA CLASSE
            first_class = cim_lyr.colorizer.classBreaks[0]
            
            # Upper value = 0
            first_class.upperBound = 0
            
            # Remover label
            first_class.label = ""
            
            # No color / transparente
            first_class.color.values = [255, 255, 255, 0]
            
            # SEGUNDA CLASSE
            second_class = cim_lyr.colorizer.classBreaks[1]
            
            # Limite superior da segunda classe
            upper2 = round(second_class.upperBound, 3)
            
            # Texto formatado com vírgula
            upper2_txt = f"{upper2:.3f}".replace(".", ",")
            
            # Novo label começando em 0
            second_class.label = f"0,001 - {upper2_txt}"
                        
            # Aplicar alterações CIM
            lyr.setDefinition(cim_lyr)
        
        # Transparência da camada
        lyr.transparency = 30

print("✅ Simbologia aplicada a todos os rasters.")

Aplicando simbologia em: Lavagem_2023_Area_Sul
Aplicando simbologia em: Lavagem_2023_Area_Oeste
Aplicando simbologia em: Lavagem_2023_Incra_8
Aplicando simbologia em: Lavagem_2023_Sol_Nascente_e_Por_do_Sol
Aplicando simbologia em: Lavagem_2023_Arniqueira
Aplicando simbologia em: Lavagem_2023_Aguas_Claras
Aplicando simbologia em: Lavagem_2023_Taguatinga
Aplicando simbologia em: Lavagem_2023_Brazlandia
Aplicando simbologia em: Lavagem_2023_Gama
Aplicando simbologia em: Lavagem_2023_Recanto_das_Emas
Aplicando simbologia em: Lavagem_2023_Santa_Maria
Aplicando simbologia em: Lavagem_2023_Riacho_Fundo_II
Aplicando simbologia em: Lavagem_2023_Riacho_Fundo
Aplicando simbologia em: Lavagem_2023_Samambaia
Aplicando simbologia em: Lavagem_2023_Ceilandia
✅ Simbologia aplicada a todos os rasters.


In [18]:
# Atribuindo Simbologia

In [23]:
# Dicionário com as Camadas de 2023
layers_2023 = {}

# Localizar todas as layers de 2023
for lyr in mapa.listLayers():
    
    if lyr.isRasterLayer and lyr.name.startswith("Lavagem_2023"):
        
        partes = lyr.name.split("_")
        
        # Nome da RA
        ra_nome = "_".join(partes[2:])
        
        # Guardar layer base
        layers_2023[ra_nome] = lyr

# Aplicar Simbologia
for lyr in mapa.listLayers():
    
    # Garantir que é raster
    if not lyr.isRasterLayer:
        continue
    
    # Ignorar os próprios rasters de 2023
    if lyr.name.startswith("Lavagem_2023"):
        continue
    
    nome_layer = lyr.name
    
    partes = nome_layer.split("_")
    
    # Garantir nome válido
    if len(partes) < 3:
        continue
    
    # Extrair RA
    ra_nome = "_".join(partes[2:])
    
    # Verificar se existe layer base correspondente
    if ra_nome not in layers_2023:        
        print(f"⚠️ Sem layer 2023 correspondente para: {lyr.name}")
        continue

    # Layer Referênca    
    source_lyr = layers_2023[ra_nome]
    
    # CIM origem
    source_cim = source_lyr.getDefinition('V3')
    
    # CIM destino
    target_cim = lyr.getDefinition('V3')
    
    # Copiar Colorizer Completo
    
    target_cim.colorizer = copy.deepcopy(source_cim.colorizer)
    
    # Aplicar definição
    lyr.setDefinition(target_cim)
    
    # Transparência da layer
    lyr.transparency = source_lyr.transparency

print("🎨 Processo finalizado com sucesso.")

🎨 Processo finalizado com sucesso.


In [25]:
# Exportando Layout - AOI (RAs)

In [24]:
# Caminho do projeto
aprx_path = r"F:\Projetos_ArcGIS_Gama\Lavagem\Lavagem.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Layout e Pasta de Saída dos Mapas
layout_model = aprx.listLayouts("A1 - LAVAGEM")[0]
out_folder = r"F:\Projetos_ArcGIS_Gama\Lavagem\Mapas"

# Mapa e Map Frame do Layout
mapa_principal = aprx.listMaps("Mapa_Principal")[0]
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Textos Dinâmicos
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]
ano_selo = layout_model.listElements("TEXT_ELEMENT", "Ano")[0]
ano_caps = layout_model.listElements("TEXT_ELEMENT", "Ano_Caps")[0]

# Bookmarks
bookmarks = mapa_principal.listBookmarks()

# Camadas que serão reativadas
camadas_urbanismo = ["Via_por_AOI_v4", "Quadras_Label_Mapas_de_Calor"]

# Loop
for bm in bookmarks:
    
    nome_ra = bm.name
    print(f"\n📍 Processando RA: {nome_ra}")
    
    # Encontrar todos os rasters da RA
    rasters_ra = [
        lyr for lyr in mapa_principal.listLayers()
        if lyr.isRasterLayer
        and lyr.name.startswith("Lavagem_")
        and nome_ra in lyr.name
    ]
    
    for raster_layer in rasters_ra:
        
        # Extrair ano do nome
        partes = raster_layer.name.split("_")
        ano = partes[1]
        
        # Desligar todas as camadas
        for lyr in mapa_principal.listLayers():
            lyr.visible = False
        
        # Ligar raster do ano atual
        raster_layer.visible = True
        
        # Urbanismo + filtro + selo
        selo_texto = nome_ra
        
        for urb_name in camadas_urbanismo:
            
            urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
            
            if urb_layer:
                urb_layer = urb_layer[0]
                urb_layer.visible = True
                urb_layer.definitionQuery = f"RA_Python = '{nome_ra}'"
                
                # Buscar selo
                if urb_name == "Via_por_AOI_v4":
                    with arcpy.da.SearchCursor(
                        urb_layer.dataSource,
                        ["RA_Python", "Selo"]
                    ) as cursor:
                        for row in cursor:
                            if row[0] == nome_ra:
                                selo_texto = row[1]
                                break
        
        # Atualizar textos no layout
        selo_elemento.text = selo_texto
        ano_selo.text = ano
        ano_caps.text = ano
        
        # Aplicar bookmark
        map_frame.zoomToBookmark(bm)
        
        # Nome do arquivo com RA + ano
        out_png = os.path.join(
            out_folder,
            f"{nome_ra}_{ano}.png"
        )
        
        layout_model.exportToPNG(out_png, resolution=300)

print("\n🏁 Processo de exportação finalizado.")


📍 Processando RA: Gama

📍 Processando RA: Aguas_Claras

📍 Processando RA: Taguatinga

📍 Processando RA: Brazlandia

📍 Processando RA: Samambaia

📍 Processando RA: Sol_Nascente_e_Por_do_Sol

📍 Processando RA: Ceilandia

📍 Processando RA: Santa_Maria

📍 Processando RA: Recanto_das_Emas

📍 Processando RA: Riacho_Fundo_II

📍 Processando RA: Riacho_Fundo

📍 Processando RA: Arniqueira

📍 Processando RA: Incra_8

🏁 Processo de exportação finalizado.


In [3]:
# Exportando Layout - Área Sul e Oeste

In [2]:
# Caminho do projeto
aprx_path = r"F:\Projetos_ArcGIS_Gama\Lavagem\Lavagem.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Layout e Pasta de Saída dos Mapas
layout_model = aprx.listLayouts("A1 - Lavagem")[0]
out_folder = r"F:\Projetos_ArcGIS_Gama\Lavagem\Mapas"

# Mapa e Map Frame do Layout
mapa_principal = aprx.listMaps("Mapa_Principal")[0]
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Textos Dinâmicos
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]
ano_selo = layout_model.listElements("TEXT_ELEMENT", "Ano")[0]
ano_caps = layout_model.listElements("TEXT_ELEMENT", "Ano_Caps")[0]

# Bookmarks
bookmarks = mapa_principal.listBookmarks()

# Camadas que serão reativadas
camadas_urbanismo = ["Via_por_AOI_v4_Sul_Oeste", "Quadras_Label_Mapas_de_Calor_Sul_Oeste"]

# Loop
for bm in bookmarks:
    
    nome_ra = bm.name
    print(f"\n📍 Processando RA: {nome_ra}")
    
    # Encontrar todos os rasters da RA
    rasters_ra = [
        lyr for lyr in mapa_principal.listLayers()
        if lyr.isRasterLayer
        and lyr.name.startswith("Lavagem_")
        and nome_ra in lyr.name
    ]
    
    for raster_layer in rasters_ra:
        
        # Extrair ano do nome
        partes = raster_layer.name.split("_")
        ano = partes[1]
        
        # Desligar todas as camadas
        for lyr in mapa_principal.listLayers():
            lyr.visible = False
        
        # Ligar raster do ano atual
        raster_layer.visible = True
        
        # Urbanismo + filtro + selo
        selo_texto = nome_ra
        
        for urb_name in camadas_urbanismo:
            
            urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
            
            if urb_layer:
                urb_layer = urb_layer[0]
                urb_layer.visible = True
                urb_layer.definitionQuery = f"RA_Python = '{nome_ra}'"
                
                # Buscar selo
                if urb_name == "Via_por_AOI_v4_Sul_Oeste":
                    with arcpy.da.SearchCursor(
                        urb_layer.dataSource,
                        ["RA_Python", "Selo"]
                    ) as cursor:
                        for row in cursor:
                            if row[0] == nome_ra:
                                selo_texto = row[1]
                                break
        
        # Atualizar textos no layout
        selo_elemento.text = selo_texto
        ano_selo.text = ano
        ano_caps.text = ano
        
        # Aplicar bookmark
        map_frame.zoomToBookmark(bm)
        
        # Nome do arquivo com RA + ano
        out_png = os.path.join(
            out_folder,
            f"{nome_ra}_{ano}.png"
        )
        
        layout_model.exportToPNG(out_png, resolution=700)

print("\n🏁 Processo de exportação finalizado.")


📍 Processando RA: Area_Sul

📍 Processando RA: Area_Oeste

🏁 Processo de exportação finalizado.


FIM